In [2]:
# getting required packages
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [7]:
#getting the csv with state IDs. This should be in the same folder as the one you're running your scraper in. 

state_ids = pd.read_csv("state_ids.csv")

In [8]:
#reading state_ids:
state_ids.head()
len(state_ids)
state_ids
# 28 states + 8 union territories

,state_id,state_name,file_name,records,link
0,1,Andaman and Nicobar Islands,andaman_and_nicobar_islands,0,https://awbi.gov.in/Cruelty?stateid=1&complain...
1,2,Andhra Pradesh,andhra_pradesh,49,https://awbi.gov.in/Cruelty?stateid=2&complain...
2,3,Arunachal Pradesh,arunachal_pradesh,4,https://awbi.gov.in/Cruelty?stateid=3&complain...
3,4,Assam,assam,12,https://awbi.gov.in/Cruelty?stateid=4&complain...
4,5,Bihar,bihar,19,https://awbi.gov.in/Cruelty?stateid=5&complain...
5,6,Chandigarh,chandigarh,5,https://awbi.gov.in/Cruelty?stateid=6&complain...
6,7,Chhattisgarh,chhattisgarh,5,https://awbi.gov.in/Cruelty?stateid=7&complain...
7,8,Delhi,delhi,67,https://awbi.gov.in/Cruelty?stateid=8&complain...
8,9,Goa,goa,0,https://awbi.gov.in/Cruelty?stateid=9&complain...
9,10,Gujarat,gujarat,41,https://awbi.gov.in/Cruelty?stateid=10&complai...


In [9]:
# looping through each row in the dataframe with iterrows(), ignoring index (iterrows returns (index, row)) (this is for later)
for _, state in state_ids.iterrows():
    print(state["state_id"], state["state_name"]) #to see if loop works

1 Andaman and Nicobar Islands
2 Andhra Pradesh
3 Arunachal Pradesh
4 Assam
5 Bihar
6 Chandigarh
7 Chhattisgarh
8 Delhi
9 Goa
10 Gujarat
11 Haryana
12 Himachal Pradesh
13 Jammu and Kashmir
14 Jharkhand
15 Karnataka
16 Kerala
17 Ladakh
18 Lakshadweep
19 Madhya Pradesh
20 Maharashtra
21 Manipur
22 Meghalaya
23 Mizoram
24 Nagaland
25 Odisha
26 Puducherry
27 Punjab
28 Rajasthan
29 Sikkim
30 Tamil Nadu
31 Telangana
32 Dadra and Nagar Haveli and Daman and Diu
33 Tripura
34 Uttarakhand
35 Uttar Pradesh
36 West Bengal


In [10]:
#Making function to run scraper

def scrape_one_page(state_id, page, limit=100):
    url = f"https://awbi.gov.in/Cruelty?stateid={state_id}&complaint_number=&cruelty_grievance_type=&complaint_year=&limit={limit}&page={page}"
    response = requests.get(url)
    doc = BeautifulSoup(response.text, "html.parser")
    
    table = doc.select("table.table-hover")[0]
    table_rows = table.select("tr")
    
    complaints = []

    for row in table_rows[1:]:
        cells = row.select("td")
        
        if len(cells) < 11:
            continue #if you find that number of columns < 11 (reached empty table/page), skip the rest of this iteration and move on to the next row. The moment Python sees continue, it jumps back to the top of the for loop.
        complaint = {
            "sr_no": cells[0].get_text(strip=True),
            "date_of_complaint": cells[1].get_text(strip=True),
            "complaint_number": cells[2].get_text(strip=True),
            "complainant_name_org": cells[3].get_text(strip=True),
            "category_of_complaint": cells[4].get_text(strip=True),
            "complaint": cells[5].get_text(strip=True),
            "place_of_incident": cells[6].get_text(strip=True),
            "violation_of_law": cells[7].get_text(strip=True),
            "action_initiated": cells[8].get_text(strip=True),
            "status": cells[9].get_text(strip=True),
            "date_of_action_taken": cells[10].get_text(strip=True),
            "state_id": state_id,
            "page": page
        }

        complaints.append(complaint)

    return complaints

In [29]:
#testing scraper
test_rows = scrape_one_page(state_id=8, page=1)
len(test_rows)
# scraper for doing one page of Delhi worked!

67

**append() vs extend()**
* append(x): Adds x as ONE item to the list.
    [1, 2].append([3, 4])  →  [1, 2, [3, 4]]
  
* extend(xs): Adds EACH element of xs individually.
   [1, 2].extend([3, 4])  →  [1, 2, 3, 4]

* Here, scrape_one_page() returns a list of complaints for one page. We use extend() so that complaints from every page are added into one continuous list, rather than creating a nested list of pages.

In [31]:
#making page loop: function that knows how to keep asking for the next page until there are no more. testing it with one state with multiple pages.

all_rows = [] #ideally want complaints from ALL pages here. biggest bucket of complaints.
page = 1 #setting counter

while True: #Just keep going until I tell you to stop.
    page_rows = scrape_one_page(
        state_id=20, page=page
    )
    #first scrape with page = 1

    if len(page_rows) == 0: #Did my function return zero complaints (did my page have no complaints?), if yes, break. if not, go on.
        break

    all_rows.extend(page_rows) #add/append page level complaints to biggest buckets.

    print(f"Finished page {page}: {len(page_rows)} rows") #finishes, tells u what happened

    page += 1 #start loop again with page = prev number + 1 (as while true said go on forever till I break you!

Finished page 1: 100 rows
Finished page 2: 73 rows


In [11]:
#adding state ID loop and running scraper on entire page!

for _, state in state_ids[
    (state_ids["state_id"] >= 16) &
    (state_ids["state_id"] <= 36)
].iterrows(): #OUTER LOOP FOR STATE IDs
    
    all_rows = [] #reset bucket
    page = 1 #reset page ID

    while True: #INNER LOOP FOR PAGE NUMBERS (THIS IS ALSO A WAY OF MAKING LOOPS!) #while there are more pages
        page_rows = scrape_one_page(
            state_id=state["state_id"], #Keep looping on state ID and coming back here. 
            page=page
        )
        
        if len(page_rows) == 0:
            break
            
        all_rows.extend(page_rows)
        
        page += 1

    df = pd.DataFrame(all_rows)
    filename = state["file_name"]
    df.to_csv(f"{filename}.csv", index=False) # Save CSV

    #summary stats
    print()
    print("-" * 40)
    print(f"Finished: {state['state_name']}")
    print(f"State ID: {state['state_id']}")
    print(f"Pages scraped: {page - 1}")
    print(f"Total complaints: {len(all_rows)}")
    print(f"File saved: {filename}")
    print("-" * 40)
    print()
    
    time.sleep(5) #waiting for 5 seconds


----------------------------------------
Finished: Kerala
State ID: 16
Pages scraped: 1
Total complaints: 27
File saved: kerala
----------------------------------------


----------------------------------------
Finished: Ladakh
State ID: 17
Pages scraped: 0
Total complaints: 0
File saved: ladakh
----------------------------------------


----------------------------------------
Finished: Lakshadweep
State ID: 18
Pages scraped: 1
Total complaints: 1
File saved: lakshadweep
----------------------------------------


----------------------------------------
Finished: Madhya Pradesh
State ID: 19
Pages scraped: 1
Total complaints: 28
File saved: madhya_pradesh
----------------------------------------


----------------------------------------
Finished: Maharashtra
State ID: 20
Pages scraped: 2
Total complaints: 173
File saved: maharashtra
----------------------------------------


----------------------------------------
Finished: Manipur
State ID: 21
Pages scraped: 0
Total complaints: 0
